# CLAMBER Judge Upgrade: gemini-2.5-flash vs gemini-3.1-pro-preview

Compares judge accuracy on the 200-row CLAMBER eval set (100 EPISTEMIC + 100 ALEATORIC, seed=42).

**Verdict: UPGRADE** — Pro model improves overall accuracy by +4.0%

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from IPython.display import display

matplotlib.use('Agg')
sns.set_theme(style='whitegrid', palette='muted')

ROOT        = Path('../../').resolve()
OUTPUTS     = ROOT / 'outputs' / 'clamber'
FIGURES_DIR = ROOT / 'notebooks' / 'clamber' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

EVAL_INPUT   = OUTPUTS / 'clamber_eval_input.csv'
FLASH_CSV    = OUTPUTS / 'clamber_judge_eval_fewshot.csv'
PRO_CSV      = OUTPUTS / 'clamber_judge_eval_fewshot_gemini-3.1-pro-preview.csv'

FLASH_LABEL = 'gemini-2.5-flash'
PRO_LABEL   = 'gemini-3.1-pro-preview'
COLORS      = {FLASH_LABEL: 'steelblue', PRO_LABEL: 'darkorange'}

print('Root:', ROOT)
print('Flash CSV exists:', FLASH_CSV.exists())
print('Pro CSV exists:  ', PRO_CSV.exists())

Root: D:\final_project\pilot_study
Flash CSV exists: True
Pro CSV exists:   True


## 1. Load data & compute accuracy

In [2]:
def compute_accuracy(eval_df, results_csv):
    res = pd.read_csv(results_csv)
    merged = eval_df.merge(
        res[['id', 'label']].rename(columns={'id': 'eval_id'}),
        on='eval_id', how='left',
    )
    merged['pred']    = merged['label'].str.strip().str.upper()
    merged['valid']   = merged['pred'].isin({'EPISTEMIC', 'ALEATORIC'})
    merged['correct'] = merged['true_label'] == merged['pred']
    valid = merged[merged['valid']]
    per_class = valid.groupby('true_label')['correct'].mean().to_dict()
    per_sub   = valid.groupby(['true_label', 'subclass'])['correct'].mean().to_dict()
    return {
        'total':     len(merged),
        'valid':     len(valid),
        'correct':   int(valid['correct'].sum()),
        'accuracy':  valid['correct'].mean(),
        'per_class': per_class,
        'per_sub':   per_sub,
        'merged':    merged,
    }

eval_df     = pd.read_csv(EVAL_INPUT)
flash_stats = compute_accuracy(eval_df, FLASH_CSV)
pro_stats   = compute_accuracy(eval_df, PRO_CSV)

print(f'Eval set: {len(eval_df)} rows')
print(f'Flash: {flash_stats["correct"]}/{flash_stats["valid"]} correct  ({flash_stats["accuracy"]:.1%})')
print(f'Pro:   {pro_stats["correct"]}/{pro_stats["valid"]} correct  ({pro_stats["accuracy"]:.1%})')

Eval set: 200 rows
Flash: 189/200 correct  (94.5%)
Pro:   197/200 correct  (98.5%)


## 2. Summary table

In [3]:
rows = []
rows.append({
    'Metric': 'Overall', 'Subclass': '',
    FLASH_LABEL: flash_stats['accuracy'],
    PRO_LABEL:   pro_stats['accuracy'],
    'Delta':     pro_stats['accuracy'] - flash_stats['accuracy'],
})
for cls in ['EPISTEMIC', 'ALEATORIC']:
    rows.append({
        'Metric': cls, 'Subclass': '',
        FLASH_LABEL: flash_stats['per_class'].get(cls, float('nan')),
        PRO_LABEL:   pro_stats['per_class'].get(cls, float('nan')),
        'Delta':     pro_stats['per_class'].get(cls, float('nan')) - flash_stats['per_class'].get(cls, float('nan')),
    })

all_keys = sorted(set(flash_stats['per_sub']) | set(pro_stats['per_sub']))
for (label, sub) in all_keys:
    f_a = flash_stats['per_sub'].get((label, sub), float('nan'))
    p_a = pro_stats['per_sub'].get((label, sub), float('nan'))
    rows.append({'Metric': f'  {label}', 'Subclass': sub,
                 FLASH_LABEL: f_a, PRO_LABEL: p_a, 'Delta': p_a - f_a})

summary_df = pd.DataFrame(rows)
display_df = summary_df.copy()
for col in [FLASH_LABEL, PRO_LABEL]:
    display_df[col] = display_df[col].apply(lambda v: f'{v:.1%}' if not np.isnan(v) else '-')
display_df['Delta'] = display_df['Delta'].apply(lambda v: f'{v:+.1%}' if not np.isnan(v) else '-')
display(display_df.set_index(['Metric', 'Subclass']))

gemini-2.5-flash gemini-3.1-pro-preview   Delta
Metric      Subclass                                                    
Overall                             94.5%                  98.5%   +4.0%
EPISTEMIC                          100.0%                  99.0%   -1.0%
ALEATORIC                           89.0%                  98.0%   +9.0%
  ALEATORIC co-reference           100.0%                 100.0%   +0.0%
            polysemy               100.0%                 100.0%   +0.0%
            what                    78.9%                  94.7%  +15.8%
            when                    86.7%                 100.0%  +13.3%
            where                   91.3%                 100.0%   +8.7%
            whom                    83.3%                  94.4%  +11.1%
  EPISTEMIC ICL                    100.0%                 100.0%   +0.0%
            NK                     100.0%                  98.0%   -2.0%

## 3. Overall accuracy bar chart

In [4]:
fig, ax = plt.subplots(figsize=(7, 4))
models = [FLASH_LABEL, PRO_LABEL]
accs   = [flash_stats['accuracy'], pro_stats['accuracy']]
colors = [COLORS[m] for m in models]
bars   = ax.bar(models, accs, color=colors, width=0.4, edgecolor='white', linewidth=1.5)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, acc + 0.005,
            f'{acc:.1%}', ha='center', va='bottom', fontsize=13, fontweight='bold')
ax.set_ylim(0.85, 1.02)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('CLAMBER Judge Accuracy - Overall (200 rows)', fontsize=13)
ax.axhline(flash_stats['accuracy'], color=COLORS[FLASH_LABEL], linestyle='--', linewidth=1, alpha=0.4)
sns.despine()
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'judge_overall_accuracy.png', dpi=150)
plt.show()
print('Saved: judge_overall_accuracy.png')

Saved: judge_overall_accuracy.png


C:\Users\dagxx\AppData\Local\Temp\ipykernel_33664\2233370494.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Per-class accuracy grouped bar chart

In [5]:
classes   = ['EPISTEMIC', 'ALEATORIC']
flash_cls = [flash_stats['per_class'].get(c, 0) for c in classes]
pro_cls   = [pro_stats['per_class'].get(c, 0)   for c in classes]
x, w      = np.arange(len(classes)), 0.32
fig, ax   = plt.subplots(figsize=(7, 4))
b1 = ax.bar(x - w/2, flash_cls, w, label=FLASH_LABEL, color=COLORS[FLASH_LABEL], edgecolor='white')
b2 = ax.bar(x + w/2, pro_cls,   w, label=PRO_LABEL,   color=COLORS[PRO_LABEL],   edgecolor='white')
for bar, val in zip(list(b1) + list(b2), flash_cls + pro_cls):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.004,
            f'{val:.1%}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(classes, fontsize=12)
ax.set_ylim(0.85, 1.05)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Per-class Judge Accuracy', fontsize=13)
ax.legend(fontsize=10)
sns.despine()
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'judge_per_class_accuracy.png', dpi=150)
plt.show()
print('Saved: judge_per_class_accuracy.png')

Saved: judge_per_class_accuracy.png


C:\Users\dagxx\AppData\Local\Temp\ipykernel_33664\3601098527.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Per-subclass accuracy bar chart

In [6]:
all_keys   = sorted(set(flash_stats['per_sub']) | set(pro_stats['per_sub']))
sub_labels = [f'{lbl} / {sub}' for lbl, sub in all_keys]
flash_vals = [flash_stats['per_sub'].get(k, float('nan')) for k in all_keys]
pro_vals   = [pro_stats['per_sub'].get(k, float('nan'))   for k in all_keys]
deltas     = [p - f for p, f in zip(pro_vals, flash_vals)]
x, w       = np.arange(len(sub_labels)), 0.32
fig, ax    = plt.subplots(figsize=(12, 4))
b1 = ax.bar(x - w/2, flash_vals, w, label=FLASH_LABEL, color=COLORS[FLASH_LABEL], edgecolor='white')
b2 = ax.bar(x + w/2, pro_vals,   w, label=PRO_LABEL,   color=COLORS[PRO_LABEL],   edgecolor='white')
for xi, (fv, pv, d) in enumerate(zip(flash_vals, pro_vals, deltas)):
    col = 'green' if d > 0 else ('red' if d < 0 else 'gray')
    ax.text(xi, max(fv, pv) + 0.025, f'{d:+.0%}',
            ha='center', va='bottom', fontsize=8, color=col, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(sub_labels, rotation=35, ha='right', fontsize=9)
ax.set_ylim(0.7, 1.12)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('Per-subclass Judge Accuracy  (delta = Pro - Flash)', fontsize=12)
ax.legend(fontsize=10)
sns.despine()
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'judge_per_subclass_accuracy.png', dpi=150)
plt.show()
print('Saved: judge_per_subclass_accuracy.png')

Saved: judge_per_subclass_accuracy.png


C:\Users\dagxx\AppData\Local\Temp\ipykernel_33664\3817237942.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Confusion matrices side-by-side

In [7]:
LABELS = ['EPISTEMIC', 'ALEATORIC']

def make_cm(merged_df):
    """Build a 2x2 confusion matrix without sklearn."""
    valid = merged_df[merged_df['valid']].copy()
    cm = np.zeros((2, 2), dtype=int)
    for i, true_lbl in enumerate(LABELS):
        for j, pred_lbl in enumerate(LABELS):
            cm[i, j] = int(((valid['true_label'] == true_lbl) & (valid['pred'] == pred_lbl)).sum())
    return cm

def plot_cm(ax, merged_df, title):
    cm      = make_cm(merged_df)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(
        cm_norm, annot=True, fmt='.1%', cmap='Blues',
        xticklabels=LABELS, yticklabels=LABELS,
        vmin=0, vmax=1, ax=ax, cbar=False,
    )
    for i in range(2):
        for j in range(2):
            ax.text(j + 0.5, i + 0.72, f'n={cm[i, j]}',
                    ha='center', va='center', fontsize=8, color='black')
    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('True', fontsize=10)
    ax.set_title(title, fontsize=11)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_cm(axes[0], flash_stats['merged'],
        f'gemini-2.5-flash  ({flash_stats["accuracy"]:.1%})')
plot_cm(axes[1], pro_stats['merged'],
        f'gemini-3.1-pro-preview  ({pro_stats["accuracy"]:.1%})')
plt.suptitle('Confusion Matrices - CLAMBER 200-row Eval', fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'judge_confusion_matrices.png', dpi=150)
plt.show()
print('Saved: judge_confusion_matrices.png')

Saved: judge_confusion_matrices.png


C:\Users\dagxx\AppData\Local\Temp\ipykernel_33664\1995997399.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Errors fixed vs regressions

In [8]:
flash_m = flash_stats['merged'].copy()
pro_m   = pro_stats['merged'].copy()
flash_m['flash_correct'] = flash_m['valid'] & flash_m['correct']
flash_m['flash_pred']    = flash_m['pred']
pro_m['pro_correct']     = pro_m['valid'] & pro_m['correct']
pro_m['pro_pred']        = pro_m['pred']

combined = flash_m[['eval_id', 'clarifying_question', 'true_label', 'subclass',
                     'flash_correct', 'flash_pred']].merge(
    pro_m[['eval_id', 'pro_correct', 'pro_pred']], on='eval_id'
)

fixed      = combined[~combined['flash_correct'] &  combined['pro_correct']]
regress    = combined[ combined['flash_correct'] & ~combined['pro_correct']]
both_wrong = combined[~combined['flash_correct'] & ~combined['pro_correct']]

print(f'Flash errors:       {(~combined["flash_correct"]).sum()}')
print(f'Pro errors:         {(~combined["pro_correct"]).sum()}')
print(f'Errors fixed:       {len(fixed)}')
print(f'Regressions (new):  {len(regress)}')
print(f'Both still wrong:   {len(both_wrong)}')

Flash errors:       11
Pro errors:         3
Errors fixed:       9
Regressions (new):  1
Both still wrong:   2


In [9]:
print('--- ERRORS FIXED BY UPGRADE (flash wrong -> pro right) ---')
for _, row in fixed.iterrows():
    print(f'  [{row["true_label"]:>9}] [{row["subclass"]:>12}]  flash={row["flash_pred"]}')
    print(f'    {str(row["clarifying_question"])[:110]}')

if len(regress):
    print('\n--- REGRESSIONS (flash right -> pro wrong) ---')
    for _, row in regress.iterrows():
        print(f'  [{row["true_label"]:>9}] [{row["subclass"]:>12}]  pro={row["pro_pred"]}')
        print(f'    {str(row["clarifying_question"])[:110]}')
else:
    print('\nNo regressions.')

if len(both_wrong):
    print('\n--- STILL WRONG IN BOTH MODELS ---')
    for _, row in both_wrong.iterrows():
        print(f'  [{row["true_label"]:>9} -> flash={row["flash_pred"]:>9}, pro={row["pro_pred"]:>9}] [{row["subclass"]:>12}]')
        print(f'    {str(row["clarifying_question"])[:110]}')

--- ERRORS FIXED BY UPGRADE (flash wrong -> pro right) ---
  [ALEATORIC] [        what]  flash=EPISTEMIC
    which one dies: person injured by axe, dog that first dies, or dog on top of old dan's grave
  [ALEATORIC] [        whom]  flash=EPISTEMIC
    What is your current fitness level and running experience?
  [ALEATORIC] [        what]  flash=EPISTEMIC
    What is the list?
  [ALEATORIC] [       where]  flash=EPISTEMIC
     Which place did the new star wars movie come out: Shrine Auditorium, in the United States, or at the Royal Al
  [ALEATORIC] [        when]  flash=EPISTEMIC
     Which period was it released: announced, or at the Geneva Motor Show?
  [ALEATORIC] [        whom]  flash=EPISTEMIC
    What are the interests and preferences of the 12 year old boy?
  [ALEATORIC] [       where]  flash=EPISTEMIC
    Which place is the episode being released: Japan, or the United States?
  [ALEATORIC] [        what]  flash=EPISTEMIC
     In which one: gland, or cell?
  [ALEATORIC] [        

## 8. Final verdict

In [10]:
delta   = pro_stats['accuracy'] - flash_stats['accuracy']
verdict = 'UPGRADE' if delta > 0 else ('TIE' if delta == 0 else 'NO IMPROVEMENT')

print('=' * 60)
print('VERDICT:', verdict)
print('=' * 60)
print(f'  gemini-2.5-flash:          {flash_stats["accuracy"]:.1%}  ({flash_stats["correct"]}/{flash_stats["valid"]} correct)')
print(f'  gemini-3.1-pro-preview:    {pro_stats["accuracy"]:.1%}  ({pro_stats["correct"]}/{pro_stats["valid"]} correct)')
print(f'  Delta:                    {delta:+.1%}')
print(f'  Errors fixed:              {len(fixed)}')
print(f'  Regressions:               {len(regress)}')
print()
if verdict == 'UPGRADE':
    print('  -> REPLACE gemini-2.5-flash with gemini-3.1-pro-preview as judge.')
else:
    print('  -> Keep gemini-2.5-flash (faster + cheaper).')
print('=' * 60)

VERDICT: UPGRADE
  gemini-2.5-flash:          94.5%  (189/200 correct)
  gemini-3.1-pro-preview:    98.5%  (197/200 correct)
  Delta:                    +4.0%
  Errors fixed:              9
  Regressions:               1

  -> REPLACE gemini-2.5-flash with gemini-3.1-pro-preview as judge.
